# Regression



In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Set plotting style
sns.set_theme(style='whitegrid')
print('Setup complete. Libraries imported successfully.')

Setup complete. Libraries imported successfully.


**Loading and Preprocessing**

In [5]:
from sklearn.datasets import fetch_california_housing

# Fetch the dataset
california = fetch_california_housing(as_frame=True)

# Extract features and target into a single DataFrame
df = california.frame

# Display basic information about the dataset
print("Dataset description and shape:")
print(f"Shape: {df.shape}\n")
df.info()

Dataset description and shape:
Shape: (20640, 9)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [3]:
# Display the first 5 rows
display(df.head())

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [6]:
# Check for missing values in each column
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

Missing values per column:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64


In [7]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

# Initialize and fit the scaler on features
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)

# Convert back to DataFrame for ease of use
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

print("Features scaled successfully. Sample of scaled features:")
display(X_scaled.head())

Features scaled successfully. Sample of scaled features:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,2.344766,0.982143,0.628559,-0.153758,-0.974429,-0.049597,1.052548,-1.327835
1,2.332238,-0.607019,0.327041,-0.263336,0.861439,-0.092512,1.043185,-1.322844
2,1.782699,1.856182,1.155620,-0.049016,-0.820777,-0.025843,1.038503,-1.332827
3,0.932968,1.856182,0.156966,-0.049833,-0.766028,-0.050329,1.038503,-1.337818
4,-0.012881,1.856182,0.344711,-0.032906,-0.759847,-0.085616,1.038503,-1.337818


 Explanation and Justification of Preprocessing Steps

 A. Handling Missing Values
* **What we did:** We checked for null or missing values across all columns and confirmed there are none ($0$ missing values).
* **Justification:** Machine learning algorithms in `scikit-learn` generally cannot handle missing values (`NaN` or `None`) and will throw errors during training. Verifying this step ensures we do not need imputation strategies (like replacing missing values with the mean or median) before proceeding.

#### B. Feature Scaling (Standardization / Z-score Normalization)
* **What we did:** We transformed our independent variables ($X$) using standard scaling so that each feature has a mean ($\mu$) of $0$ and a standard deviation ($\sigma$) of $1$.
* **Justification:**
  1. **Differences in Scales:** Our features have vastly different ranges. For instance, `Population` values range into the thousands, while `AveBedrms` or `MedInc` (median income) are in small single or double digits. Unscaled features with larger magnitudes can disproportionately dominate the model's loss function.
  2. **Model Stability and Convergence:** Optimization algorithms (like Gradient Descent used in some regression models) and distance-based metrics converge significantly faster and remain numerically stable when features are on the same scale.
  3. **Fair Regularization:** If we apply regularization techniques (like Ridge or Lasso), scaling is crucial to ensure that penalty terms are applied fairly across all coefficients regardless of their original unit scale.

**Regression Algorithm Implementation**

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize the models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Support Vector Regressor (SVR)": SVR(C=1.0, epsilon=0.1)
}

# Train each model and store their predictions
predictions = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)

print("\nAll models have been trained and predictions have been gathered.")

Training Linear Regression...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training Support Vector Regressor (SVR)...

All models have been trained and predictions have been gathered.


### 5. Algorithm Explanations and Suitability

#### 1. Linear Regression
* **How it works:** It models a linear relationship between the independent input features and the target variable by fitting a straight hyperplane that minimizes the Sum of Squared Errors (Residuals).
* **Suitability:** It serves as a strong, interpretable baseline. If features like `MedInc` (median income) have a strong, linear correlation with house values, Linear Regression will capture it with low computational overhead.

#### 2. Decision Tree Regressor
* **How it works:** It splits the dataset into smaller subsets based on feature thresholds, forming a tree structure. For predictions, it assigns the average target value of the training points in the leaf node where a new sample lands.
* **Suitability:** Housing data often contains non-linear patterns and complex feature interactions (such as the combined effect of latitude and longitude on price). Decision Trees naturally handle these interactions without requiring feature engineering.

#### 3. Random Forest Regressor
* **How it works:** An ensemble method that trains multiple independent decision trees on bootstrap samples of the training data (bagging) and averages their predictions.
* **Suitability:** Single decision trees are highly prone to overfitting. Random Forest reduces variance and improves generalization, making it exceptionally robust to noise and outliers present in spatial data like California Housing.

#### 4. Gradient Boosting Regressor
* **How it works:** An ensemble technique that builds trees sequentially. Each new tree is trained to correct the errors (residuals) made by the previous trees using gradient descent.
* **Suitability:** It often yields the highest predictive performance. Because geographical and socio-economic variables in housing can be highly complex, Gradient Boosting sequentially refines its predictions to capture very subtle patterns.

#### 5. Support Vector Regressor (SVR)
* **How it works:** SVR finds a hyperplane in a high-dimensional space that fits the data within a specified margin of tolerance ($\epsilon$), ignoring errors smaller than this threshold.
* **Suitability:** By utilizing kernel functions (such as RBF), SVR can effectively map non-linear relationships. This is particularly useful for modeling the complex, continuous boundary values of geographical coordinates (Latitude/Longitude) relative to price.

**Model Evaluation and Comparison**

In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

evaluation_results = []

for name, pred in predictions.items():
    mse = mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    evaluation_results.append({
        "Model": name,
        "MSE": mse,
        "MAE": mae,
        "R2 Score": r2
    })

# Convert results to DataFrame for a clean table view
df_eval = pd.DataFrame(evaluation_results)
display(df_eval)

,Model,MSE,MAE,R2 Score
0,Linear Regression,0.555892,0.533200,0.575788
1,Decision Tree,0.494272,0.453784,0.622811
2,Random Forest,0.255498,0.327613,0.805024
3,Gradient Boosting,0.293999,0.371650,0.775643
4,Support Vector Regressor (SVR),0.355198,0.397763,0.728941


###  Model Comparison & Analysis

Based on the performance metrics calculated on the test set:

| Model | MSE | MAE | R2 Score |
| :--- | :---: | :---: | :---: |
| **Linear Regression** | 0.5559 | 0.5332 | 0.5758 |
| **Decision Tree** | 0.4943 | 0.4538 | 0.6228 |
| **Support Vector Regressor (SVR)** | 0.3552 | 0.3978 | 0.7289 |
| **Gradient Boosting** | 0.2940 | 0.3717 | 0.7756 |
| **Random Forest** | **0.2555** | **0.3276** | **0.8050** |

---

#### A. The Best-Performing Algorithm: **Random Forest Regressor**
* **Justification:**
  * **Lowest Error and Highest R2:** The Random Forest Regressor achieved the highest **R2 Score of ~80.5%** and the lowest prediction errors (**MSE of 0.2555** and **MAE of 0.3276**).
  * **Robust Handling of Non-Linearity & Interactions:** Housing valuation is highly dependent on non-linear interaction terms (e.g., location coordinates *Latitude* and *Longitude* combined with income levels). By bagging multiple independent decision trees, Random Forest excels at mapping these complex interactions and reducing the high variance associated with single decision trees, resulting in excellent generalization on the unseen test set.

#### B. The Worst-Performing Algorithm: **Linear Regression**
* **Reasoning:**
  * **Highest Error and Lowest R2:** Linear Regression performed the worst, with the lowest **R2 Score (~57.6%)** and the highest **MSE (0.5559)**.
  * **Assumption of Linearity:** Linear Regression assumes a strict linear relationship between features and the target variable. However, spatial data (like geography) and complex socio-economic features in real-world housing datasets rarely exhibit purely linear patterns. For example, moving a few miles in latitude/longitude can non-linearly change home prices, a pattern that a simple linear plane cannot capture, leading to high bias.